# 05. Random Forest - Breast Cancer Diagnosis## What is this notebook about?A **Random Forest** is many decision trees trained on random slices of the data. Each tree votes on the prediction, and the majority wins.This simple idea (the "wisdom of crowds") fixes the biggest weakness of Decision Trees: their tendency to overfit.## What you will learn1. How a Random Forest improves over a single Decision Tree2. How the number of trees (`n_estimators`) affects stability3. How to read **feature importance** in a Random Forest4. How to evaluate a binary classifier with **ROC-AUC**## DatasetThe **Breast Cancer Wisconsin** dataset: 569 tumour samples, 30 features (cell measurements), and a binary target (malignant or benign). Built into sklearn.This is a classic medical-classification task and exactly the kind of problem Random Forests excel at.

In [ ]:
# If you're running locally, uncomment and run this once.# In Google Colab, all of these are pre-installed - you can skip this cell.# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
# Standard imports used in every ML notebookimport numpy as np                 # numerical arraysimport pandas as pd                # tabular data (DataFrames)import matplotlib.pyplot as plt    # plottingimport seaborn as sns              # prettier statistical plots# Set up plot stylesns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)# Set a random seed so results are reproduciblenp.random.seed(42)

In [ ]:
from sklearn.datasets import load_breast_cancerfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import (    accuracy_score, classification_report, roc_auc_score, confusion_matrix,)# Load the datasetdata = load_breast_cancer(as_frame=True)X, y = data.data, data.targetprint(f"Shape: {X.shape}")print(f"Classes: {data.target_names}")print(f"Class distribution: {np.bincount(y)}")

## Step 1. Single Tree vs Random ForestLet's compare them head to head.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)# A single decision tree (no depth limit - so it might overfit)tree   = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)# A random forest of 200 treesforest = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)# Compare them on accuracy AND ROC-AUCprint(f"{'Model':20s} {'Accuracy':>10s} {'ROC-AUC':>10s}")print("-" * 42)for name, m in [("Decision Tree", tree), ("Random Forest", forest)]:    pred = m.predict(X_test)    proba = m.predict_proba(X_test)[:, 1]    print(f"{name:20s} {accuracy_score(y_test, pred):>10.3f} {roc_auc_score(y_test, proba):>10.3f}")

**Random Forest wins** because averaging 200 trees smooths out the variance of a single tree.## Step 2. Effect of the number of treesMore trees = more stable predictions, with diminishing returns past a certain point.

In [ ]:
# Try several values of n_estimators and measure CV accuracyns = [1, 5, 10, 25, 50, 100, 200, 500]scores = []for n in ns:    rf = RandomForestClassifier(n_estimators=n, random_state=42)    s = cross_val_score(rf, X_train, y_train, cv=5, scoring="accuracy").mean()    scores.append(s)    print(f"n_estimators = {n:4d} -> CV accuracy = {s:.4f}")# Plot the curveplt.figure(figsize=(10, 5))plt.plot(ns, scores, "o-", color="black")plt.xscale("log")plt.xlabel("Number of trees (n_estimators)")plt.ylabel("CV accuracy")plt.title("More trees stabilize predictions, with diminishing returns")plt.show()

**Common practice:** start with `n_estimators=100`, then go up to 500 if you have the compute. Past that the gain is tiny.## Step 3. Feature ImportanceRandom Forests aggregate feature importance across all 200 trees, giving a robust signal.

In [ ]:
# Take only the top 15 most important features (out of 30)imp = pd.Series(forest.feature_importances_, index=X.columns).sort_values().tail(15)plt.figure(figsize=(8, 6))imp.plot.barh(color="gray")plt.title("Top 15 Most Important Features")plt.xlabel("Importance")plt.show()

**What you'll see:** measurements like `worst concave points`, `worst perimeter`, and `worst area` dominate. These describe the most extreme tumour cells observed - which makes biological sense.## Step 4. Confusion Matrix

In [ ]:
# Get predictions on the test sety_pred = forest.predict(X_test)# Plot confusion matrixcm = confusion_matrix(y_test, y_pred)sns.heatmap(cm, annot=True, fmt="d", cmap="Greys",            xticklabels=data.target_names, yticklabels=data.target_names)plt.xlabel("Predicted")plt.ylabel("Actual")plt.title("Random Forest Confusion Matrix")plt.show()# In medical contexts, false negatives (missed malignancies) are the most costly.# Look closely at the off-diagonal numbers.print("\nDetailed report:")print(classification_report(y_test, y_pred, target_names=data.target_names))

## Step 5. Exercises1. **Try `max_features="sqrt"` vs `"log2"` vs `1.0`** (use all features). The default for classification is `"sqrt"`. How does it affect the score?2. **Train a `RandomForestRegressor`** on California Housing.3. **Use the OOB score** instead of cross-validation:   ```python   rf = RandomForestClassifier(n_estimators=200, oob_score=True).fit(X, y)   print(rf.oob_score_)   ```4. **Time how long it takes** to train 1000 trees vs 100. Does it scale linearly?5. **Try a Kaggle dataset** like the Telco Customer Churn dataset.Next: **06 - Gradient Boosting & XGBoost**!